### RaschPy Differential Item Functioning (DIF) worked example

Differential Item Functioning occurs when persons with the same overall trait level, but from different groups (e.g. Gender, age or first language), have different probabilities of success on a given item, which violates measurement principles and represents a threat to the validity of scores if unaddressed. `dif_test()` (available for `SLM`, `PCM`, and `RSM`; not yet implemented for `MFRM`) tests every item for DIF against a chosen reference group:

1. Reads in a file mapping persons to exogenous variables, stored as an attribute `self.exogenous`. This file does not need to have a mapping for every person to every variable; the test ignores persons who are unlabelled for the relevant variable.
2. Splits non-extreme persons into groups by a named column in `self.exogenous` (any number of groups; each "focal" group is compared to a single "reference" group, by default the largest).
3. Calibrates each group independently, then purifies both groups onto a common scale — genuinely DIF-affected items are excluded from the scale used to detect DIF so they do not distort the test.
4. Tests every item (not only the ones used to define the purified scale) via a per-item Wald test, likelihood-ratio test, or both, plus an omnibus test that tests whether DIF exists in any items in the data. $ p $-values are corrected for multiple comparisons across items by default (BH by default, Bonferroni also available) before the flagging rule is applied.

For PCM and RSM, which have a threshold/category structure in addition to item location, `dif_test()` also separately tests for Differential Category Functioning (DCF) via `threshold_dif_table`. Here we make a distinction between DCF and the more standard DSF approach, which tests for differences in threshold locations: instead, DCF tests for differences in category *widths* between adjacent thresholds ($ \tau_{k+1} - \tau_{k} $), tested category by category, reference vs. focal. This matters because two groups can show identical overall item difficulty — nothing flagged in `dif_table` — while still using the rating scale differently, e.g. one group's raters compress two middle categories together or all but skip one; that is a real, substantive difference in how the item functions, but it is invisible to any test of the item as a whole. The issue with DSF compared to DCF is that thresholds are interdependent; with a maximum score $ m $, there are $ m $ thresholds but only $ m - 1 $ degrees of freedom, due to the sum-to-zero constraint on Rasch-Andrich thresholds. For example, with a maximum score of 2, if one threshold shifts, the other must by definition shift by an equal and opposite amount; the standard error of the middle (only unbounded) category is precisely twice that of each threshold (equal by definition). This means that a change in the functioning (width) of one category propagates through the rest of the thresholds, confounding the DSF test. DCF, on the other hand, avoids this by examining the appropriate, independent unit (with the correct degrees of freedom: $ m - 1 $ df for $ m - 1 $ bounded categories). 

`dif_test()` also optionally adds an ETS-style DIF-magnitude `Category` column (`category=True`), following Zwick, Thayer & Mazzeo (1997): each item is classified 'A' (negligible), 'B+'/'B-' (slight to moderate), or 'C+'/'C-' (moderate to large). The sign gives the direction: '+' means the item is harder for the focal group than the reference group, '-' the converse. 'B' requires |Difference| ≥ 0.43 logits plus a significant Wald/Welch $ p $-value (`category_alpha=0.05` by default); 'C' requires |Difference| ≥ 0.64 logits plus a separate, one-sided test that the difference is significantly above the B boundary itself (P(|Difference| ≤ 0.43) < 0.05), not merely significantly nonzero. This classification is independent of the plain `Flagged` column (which uses `alpha`/`logit_threshold` directly) — it exists so results can also be reported in ETS categories. It is scoped to item-location DIF (`dif_table`) only: the 0.43/0.64 logit boundaries are calibrated in the literature for item-location DIF specifically, not for DCF category-width tests.

This notebook simulates two groups (A and B) with known, deliberately injected DIF, and checks whether `dif_test()` successfully detects it.

In [1]:
import numpy as np
import pandas as pd
from raschpy import SLM, PCM, RSM
from raschpy.simulation import SLM_Sim, PCM_Sim, RSM_Sim

#### SLM

Simulate two groups of 300 persons each on the same 12 items. Group B has genuine DIF injected on three items: Item_1 (+1.0 logits, harder for B), Item_2 (-1.0 logits, easier for B), Item_3 (+0.5 logits). All other items are unaffected.

In [2]:
no_of_persons = 300
sim_a = SLM_Sim(no_of_items=12, no_of_persons=no_of_persons, item_range=2, seed=1)

items_b = sim_a.items.copy()
items_b['Item_1'] += 1.0
items_b['Item_2'] -= 1.0
items_b['Item_3'] += 0.5

sim_b = SLM_Sim(no_of_items=12, no_of_persons=no_of_persons, item_range=2,
                manual_items=items_b, manual_item_names=sim_a.item_names, seed=2)

resp_a, resp_b = sim_a.responses.copy(), sim_b.responses.copy()
resp_a.index = [f'A_{i}' for i in range(no_of_persons)]
resp_b.index = [f'B_{i}' for i in range(no_of_persons)]
responses = pd.concat([resp_a, resp_b])

exogenous = pd.DataFrame({'Group': ['A'] * no_of_persons + ['B'] * no_of_persons}, index=responses.index)

slm = SLM(responses, exogenous=exogenous)

Run the DIF test, comparing group B against reference group A:

In [3]:
slm.dif_test('Group',
             reference='A',
             selection_method='wald',   # purification method: 'wald' (default) | 'robust_z' | 'none'
             test='both',               # per-item flagging: 'wald' | 'lr' | 'both'
             omnibus=True,              # Andersen-style joint test, all items at once
             category=True,             # adds an ETS-style A/B/C DIF-magnitude category column
             correction='bh',           # multiple-comparison correction: 'bh' | 'bonferroni' | None
             welch=True,                # Welch's t-test (Satterthwaite df) instead of z, more conservative for small/unequal groups
             no_of_samples=500)

round(slm.dif_table, 3)

,Group,Reference,Focal,Focal (purified),Difference,SE,z,p,p (corrected),Selected,Flagged,df,Category,LR,p_LR,p_LR (corrected),Flagged_LR
Item,,,,,,,,,,,,,,,,,
Item_1,B,0.190,1.060,1.077,0.887,0.205,4.331,0.000,0.000,False,True,531.626,C+,18.699,0.000,0.000,True
Item_2,B,0.905,-0.154,-0.137,-1.041,0.198,-5.259,0.000,0.000,False,True,537.492,C-,24.318,0.000,0.000,True
Item_3,B,-0.197,0.388,0.405,0.602,0.192,3.137,0.002,0.007,False,True,536.457,B+,11.003,0.001,0.004,True
Item_4,B,0.592,0.279,0.295,-0.296,0.193,-1.538,0.125,0.374,False,False,530.917,A,2.467,0.116,0.349,False
Item_5,B,-1.236,-1.036,-1.019,0.217,0.216,1.003,0.316,0.759,True,False,537.525,A,0.484,0.487,0.900,False
Item_6,B,-0.303,-0.383,-0.366,-0.063,0.182,-0.345,0.730,0.852,True,False,527.761,A,0.160,0.689,0.900,False
Item_7,B,0.075,0.080,0.097,0.021,0.190,0.112,0.911,0.911,True,False,535.856,A,0.009,0.925,0.925,False
Item_8,B,-0.027,-0.095,-0.079,-0.052,0.187,-0.278,0.781,0.852,True,False,537.682,A,0.049,0.825,0.900,False
Item_9,B,-0.854,-0.773,-0.757,0.097,0.204,0.477,0.633,0.852,True,False,537.995,A,0.134,0.714,0.900,False


Items 1-3, where DIF was injected, should be flagged with `Difference` estimates close to the specified 1.0 / -1.0 / 0.5 logit shifts. The remaining items should not be flagged.

In [4]:
slm.dif_omnibus_table  # is there DIF at all, across every item jointly

,LR,df,p,Flagged
B,59.000587,11,0.0,True


#### PCM

Simulate two groups on 15 4-category items (maximum score 3). Group B has the same item-location DIF pattern as before (Item_1/2/3), plus a pure category-width (DCF) shift on Item_4 only — its overall item location is left unchanged, but the width of Category 1 is increased by 0.8 logits, so it should be flagged in `threshold_dif_table` rather than `dif_table`.

In [5]:
no_of_persons = 1000
max_score_vector = [3] * 15
sim_a = PCM_Sim(no_of_items=15, no_of_persons=no_of_persons, max_score_vector=max_score_vector, item_range=2, seed=1)

items_b = sim_a.items.copy()
items_b['Item_1'] += 1.0
items_b['Item_2'] -= 1.0
items_b['Item_3'] += 0.5

# Pin thresholds to sim_a's own for every item except Item_4, which gets a
# deliberate category-width (DCF) shift with no item-location shift.
manual_thresholds = []
for item in sim_a.item_names:
    vals = sim_a.thresholds.loc[item].dropna().values.copy()
    if item == 'Item_4':
        # Isolate the shift to a single category width (w1): subtracting a
        # constant from every threshold preserves all pairwise differences, so
        # perturbing tau_1 alone and re-centering shifts w1 without touching w2
        # (unlike recomputing just the last threshold, which would couple them).
        vals[0] -= 0.8
        vals -= vals.mean()
    else:
        vals[-1] = -vals[:-1].sum()  # exact zero-sum, avoiding float residue
    manual_thresholds.append(vals)

sim_b = PCM_Sim(no_of_items=15, no_of_persons=no_of_persons, max_score_vector=max_score_vector, item_range=2,
                manual_items=items_b, manual_item_names=sim_a.item_names,
                manual_thresholds=manual_thresholds, seed=2)

resp_a, resp_b = sim_a.responses.copy(), sim_b.responses.copy()
resp_a.index = [f'A_{i}' for i in range(no_of_persons)]
resp_b.index = [f'B_{i}' for i in range(no_of_persons)]
responses = pd.concat([resp_a, resp_b])
exogenous = pd.DataFrame({'Group': ['A'] * no_of_persons + ['B'] * no_of_persons}, index=responses.index)

pcm = PCM(responses, max_score_vector=max_score_vector, exogenous=exogenous)

In [6]:
pcm.dif_test('Group',
             reference='A',
             selection_method='wald',
             test='both',
             omnibus=True,
             omnibus_scope='full',
             category=True,
             correction='bh',
             welch=True,
             no_of_samples=300)

round(pcm.dif_table, 3)   # item-location DIF

,Group,Reference,Focal,Focal (purified),Difference,SE,z,p,p (corrected),Selected,Flagged,df,Category,LR,p_LR,p_LR (corrected),Flagged_LR
Item,,,,,,,,,,,,,,,,,
Item_1,B,-0.927,0.055,0.120,1.047,0.079,13.209,0.000,0.000,False,True,1864.648,C+,231.328,0.000,0.000,True
Item_2,B,-0.310,-1.385,-1.319,-1.010,0.073,-13.904,0.000,0.000,False,True,1848.833,C-,239.389,0.000,0.000,True
Item_3,B,0.767,1.267,1.332,0.566,0.083,6.834,0.000,0.000,False,True,1975.643,B+,68.663,0.000,0.000,True
Item_4,B,-0.667,-0.565,-0.499,0.167,0.074,2.264,0.024,0.089,False,False,1977.761,A,4.837,0.028,0.104,False
Item_5,B,-0.137,-0.192,-0.126,0.011,0.067,0.166,0.868,0.883,True,False,1978.000,A,0.026,0.871,1.000,False
Item_6,B,1.007,0.980,1.045,0.038,0.071,0.536,0.592,0.880,True,False,1962.911,A,0.327,0.567,1.000,False
Item_7,B,-0.164,-0.242,-0.176,-0.012,0.065,-0.185,0.853,0.883,True,False,1964.227,A,0.000,1.000,1.000,False
Item_8,B,-0.361,-0.493,-0.428,-0.067,0.077,-0.873,0.383,0.717,True,False,1968.188,A,0.000,1.000,1.000,False
Item_9,B,1.143,0.992,1.057,-0.086,0.081,-1.062,0.289,0.618,True,False,1975.638,A,1.603,0.205,0.440,False


In [7]:
round(pcm.threshold_dif_table, 3)   # per-item category-width (DCF) test

Group  Reference  Focal  Difference     SE      z        df  \
Item    Category                                                               
Item_1  1            B      1.978  1.987       0.009  0.265  0.035  1747.323   
        2            B      1.916  1.637      -0.279  0.223 -1.255  1943.592   
Item_2  1            B      1.195  0.795      -0.400  0.297 -1.347  1753.012   
        2            B      1.086  1.230       0.144  0.235  0.613  1963.360   
Item_3  1            B      0.738  0.602      -0.136  0.217 -0.627  1969.976   
        2            B      2.039  1.784      -0.255  0.286 -0.892  1915.488   
Item_4  1            B      0.997  1.733       0.736  0.254  2.896  1975.443   
        2            B      1.909  1.792      -0.117  0.202 -0.579  1973.729   
Item_5  1            B      1.316  1.279      -0.037  0.223 -0.165  1977.969   
        2            B      1.910  1.964       0.054  0.210  0.258  1952.097   
Item_6  1            B      0.766  1.066       0.300  0.233  1.289  1966.720   
        2            B      0.932  0.720      -0.212  0.297 -0.716  1965.208   
Item_7  1            B      0.809  0.770      -0.039  0.247 -0.159  1970.319   
        2            B      1.959  1.739      -0.219  0.224 -0.979  1960.441   
Item_8  1            B      2.003  2.265       0.262  0.257  1.020  1977.790   
        2            B      1.918  1.670      -0.249  0.204 -1.218  1976.645   
Item_9  1            B      0.983  1.242       0.259  0.214  1.211  1977.609   
        2            B      1.714  1.487      -0.227  0.295 -0.771  1977.815   
Item_10 1            B      1.767  1.606      -0.161  0.266 -0.608  1975.398   
        2            B      0.997  1.272       0.275  0.210  1.307  1976.326   
Item_11 1            B      0.367  0.736       0.369  0.248  1.489  1972.863   
        2            B      0.567  0.217      -0.350  0.296 -1.182  1951.538   
Item_12 1            B      1.646  1.569      -0.076  0.238 -0.320  1973.437   
        2            B      0.603  0.099      -0.504  0.241 -2.090  1974.435   
Item_13 1            B      1.514  1.262      -0.252  0.265 -0.950  1977.548   
        2            B      1.615  1.411      -0.204  0.203 -1.004  1975.170   
Item_14 1            B      1.248  1.307       0.059  0.219  0.268  1973.728   
        2            B      1.515  1.618       0.102  0.221  0.463  1977.928   
Item_15 1            B      0.497  0.678       0.181  0.267  0.678  1977.495   
        2            B      0.270 -0.131      -0.401  0.264 -1.521  1971.413   

                      p  p (corrected)  Flagged  
Item    Category                                 
Item_1  1         0.972          0.972    False  
        2         0.210          0.419    False  
Item_2  1         0.178          0.356    False  
        2         0.540          0.540    False  
Item_3  1         0.531          0.531    False  
        2         0.373          0.531    False  
Item_4  1         0.004          0.008     True  
        2         0.563          0.563    False  
Item_5  1         0.869          0.869    False  
        2         0.796          0.869    False  
Item_6  1         0.198          0.395    False  
        2         0.474          0.474    False  
Item_7  1         0.874          0.874    False  
        2         0.328          0.656    False  
Item_8  1         0.308          0.308    False  
        2         0.223          0.308    False  
Item_9  1         0.226          0.441    False  
        2         0.441          0.441    False  
Item_10 1         0.543          0.543    False  
        2         0.191          0.383    False  
Item_11 1         0.137          0.237    False  
        2         0.237          0.237    False  
Item_12 1         0.749          0.749    False  
        2         0.037          0.074    False  
Item_13 1         0.342          0.342    False  
        2         0.315          0.342    False  
Item_14 1         0.789          0.789    False  
      

Item_4 should be flagged in `threshold_dif_table` (its category structure differs between groups) but not in `dif_table` (its item location is unaffected) — the two phenomena are deliberately kept independent in this simulation so the signal is unambiguous.

#### RSM

Simulate two groups on 20 items sharing one rating scale with 4 categories (maximum score 3). Group B again gets the Item_1/2/3 location DIF, plus the same +0.8 shift to one shared category width used for PCM's Item_4 — since RSM's thresholds (and therefore widths) are shared across all items, this is a uniform instrument-wide DCF effect rather than a single-item one.

In [8]:
no_of_persons = 1000
sim_a = RSM_Sim(no_of_items=20, no_of_persons=no_of_persons, max_score=3, item_range=2, seed=1)

items_b = sim_a.items.copy()
items_b['Item_1'] += 1.0
items_b['Item_2'] -= 1.0
items_b['Item_3'] += 0.5

thresholds_b = sim_a.thresholds.copy()
thresholds_b[1] -= 0.8
thresholds_b -= thresholds_b.mean()

sim_b = RSM_Sim(no_of_items=20, no_of_persons=no_of_persons, max_score=3, item_range=2,
                manual_items=items_b, manual_item_names=sim_a.item_names,
                manual_thresholds=thresholds_b, seed=2)

resp_a, resp_b = sim_a.responses.copy(), sim_b.responses.copy()
resp_a.index = [f'A_{i}' for i in range(no_of_persons)]
resp_b.index = [f'B_{i}' for i in range(no_of_persons)]
responses = pd.concat([resp_a, resp_b])
exogenous = pd.DataFrame({'Group': ['A'] * no_of_persons + ['B'] * no_of_persons}, index=responses.index)

rsm = RSM(responses, exogenous=exogenous)

In [9]:
rsm.dif_test('Group',
             reference='A',
             selection_method='robust_z',
             test='both',
             omnibus=True,
             category=True,
             correction='bh',
             welch=True,
             no_of_samples=300)

round(rsm.dif_table, 3)   # item-location DIF

,Group,Reference,Focal,Focal (purified),Difference,SE,z,p,p (corrected),Selected,Flagged,df,Category,LR,p_LR,p_LR (corrected),Flagged_LR
Item,,,,,,,,,,,,,,,,,
Item_1,B,-0.865,-0.030,-0.035,0.830,0.074,11.220,0.000,0.000,False,True,1991.712,C+,165.184,0.000,0.000,True
Item_2,B,-0.368,-1.294,-1.299,-0.931,0.081,-11.451,0.000,0.000,False,True,1961.016,C-,207.777,0.000,0.000,True
Item_3,B,0.624,1.146,1.141,0.517,0.077,6.730,0.000,0.000,True,True,1958.303,B+,41.017,0.000,0.000,True
Item_4,B,-0.666,-0.699,-0.704,-0.038,0.078,-0.492,0.623,0.813,True,False,1987.691,A,0.000,1.000,1.000,False
Item_5,B,-0.236,-0.229,-0.234,0.002,0.074,0.027,0.978,0.978,True,False,1992.050,A,0.011,0.917,1.000,False
Item_6,B,0.808,0.734,0.728,-0.079,0.081,-0.974,0.330,0.601,True,False,1987.002,A,0.175,0.676,1.000,False
Item_7,B,-0.325,-0.395,-0.400,-0.075,0.076,-0.997,0.319,0.601,True,False,1986.920,A,0.508,0.476,0.952,False
Item_8,B,-0.497,-0.517,-0.522,-0.026,0.075,-0.340,0.734,0.863,True,False,1986.041,A,0.000,1.000,1.000,False
Item_9,B,0.989,0.828,0.823,-0.166,0.082,-2.016,0.044,0.220,True,False,1990.127,A,4.946,0.026,0.105,False


In [10]:
rsm.dif_omnibus_table     # joint item-location test

,LR,df,p,Flagged
B,493.511357,19,0.0,True


In [11]:
round(rsm.threshold_dif_table, 3)   # shared category-width (DCF) DIF

,Group,Category,Reference,Focal,Difference,SE,z,p,p (corrected),Flagged,df
0,B,1,1.942,2.766,0.824,0.061,13.606,0.000,0.000,True,1992.899
1,B,2,0.735,0.695,-0.040,0.062,-0.649,0.516,0.516,False,1992.899


#### Simpler alternative: `andersen_lr_test(split_by='exogenous')`

For a quick two-group check without anchor purification (no per-item table, just a single joint test), use `andersen_lr_test(split_by='exogenous')`. `andersen_lr_test` is available on all three models (`SLM`, `PCM`, and `RSM`); this demo reuses the `slm` model fitted in the SLM section above (same simulated data, same injected item-location DIF on Item_1/2/3).

The check applies Andersen's (1973) likelihood-ratio test: it compares the log-likelihood of a single item calibration fit jointly across both groups against the sum of the log-likelihoods from calibrating each group separately, via `LR = -2 * (ll_joint - (ll_A + ll_B))`, with `df` equal to the number of freely-estimated item parameters that differ between the two separate calibrations. `split_by='exogenous'` splits the sample by the named covariate (`Group`) rather than the internal median split used for Andersen's test of global model fit. The check is a lighter-weight version of the `dif_test(omnibus=True)` joint test, using the same underlying LR statistic but without purification and with no per-item detail, limited to two groups. In the `SLM` example, the test rejects the null of no DIF, consistent with `dif_test()` for `SLM` above.

In [13]:
slm.andersen_lr_test(split_by='exogenous', covariate='Group')
print(f'LR = {slm.andersen_lr:.2f}, df = {slm.andersen_df}, p = {slm.andersen_p:.4f}, flagged = {slm.andersen_p<0.05}')

LR = 59.00, df = 11, p = 0.0000, flagged = True
